# 03 — HF backend generation (`transformers`)

Generating directly with **`transformers`** — load the weights, call `.generate()`. This is
the "HF backend": simpler and more hackable than vLLM (full access to logits, hidden states,
custom stopping), but slower at scale. It's the backend lm-eval uses with `--backend hf`.

**Library focus:** `transformers.AutoModelForCausalLM`, `.generate()`.

In [ ]:
MODEL = "Qwen/Qwen3.5-4B"

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="bfloat16", device_map="cuda")
model.eval()
print(model.config.architectures, "| params:", f"{model.num_parameters() / 1e9:.2f}B")

## Sampling generation

Render the chat prompt, move inputs to the model's device, and `.generate()` with sampling.
Slice off the prompt tokens before decoding so you keep only the new continuation.

In [ ]:
messages = [{"role": "user", "content": "Explain entropy to a 10-year-old."}]
prompt = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
inputs = tok(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
    )
new_tokens = out[0][inputs["input_ids"].shape[1] :]
print(tok.decode(new_tokens, skip_special_tokens=True).strip())

## Greedy (deterministic) decoding

`do_sample=False` ignores temperature/top-p and takes the argmax each step — reproducible,
the default for many eval tasks.

In [ ]:
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
print(
    tok.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()
)

## Batched generation

Batch several prompts together. **Decoder-only models must left-pad** for generation, so
set `padding_side="left"` before tokenizing a batch.

In [ ]:
tok.padding_side = "left"
batch = [
    "Q: What is 2+2? A:",
    "Q: Capital of Japan? A:",
    "Q: Color of the sky? A:",
]
enc = tok(batch, return_tensors="pt", padding=True).to(model.device)
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=16, do_sample=False)
for row, ids in zip(batch, out):
    gen = tok.decode(ids[enc["input_ids"].shape[1] :], skip_special_tokens=True)
    print(row, "->", gen.strip())

> **WSL note:** the HF backend runs on the GPU but is slower than vLLM at scale. Under
> lm-eval, `batch_size: auto` can thrash on WSL (the memory budget is over-reported) —
> prefer a *fixed* batch size (notebook 07).
>
> **Project pointer:** `scripts/evaluate.py --backend hf` drives this path through
> lm-eval's `HFLM`; `llm_core.models.resolve_profile` supplies the chat-template/thinking
> policy used when rendering prompts.